In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

news_api_key = os.getenv("NEWS_API_KEY")

In [2]:
from langchain.agents import create_agent
from newsapi import NewsApiClient

newsapi = NewsApiClient(api_key=news_api_key)

# Fetch data (searching for the supplier name)
def fetch_supplier_news(supplier_name: str) -> list[dict]:
    """Fetches news articles for a given supplier and returns a clean list of dictionaries containing only the title and description."""
    try:
        response = newsapi.get_everything(
            q=supplier_name,
            language='en',
            sort_by='relevancy',
            page_size=5
        )
        if response.get('status') == 'ok':
            articles = response.get('articles', [])
            clean_list = [
                {
                    'title': article.get('title'),
                    'description': article.get('description'),
                }
                for article in articles
            ]
            return clean_list
        else:
            print(f"Error: {response.get('message')}")
            return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []


In [3]:
fetch_supplier_news("Samsung SDI")
fetch_supplier_news("Magna International")

[{'title': 'The new bibliomaniacs',
  'description': 'Rare book collecting is booming. Young people raised in the digital age are seeking tangible connections to the past.'},
 {'title': 'TD Securities Maintains Buy Rating on Magna International (MGA)',
  'description': 'Magna International Inc. (NYSE:MGA) is one of the 10 Best Electric Vehicle Supply Chain Stocks to Invest In. On May 4, 2026, TD Securities analyst Brian...'},
 {'title': 'Is Magna International Inc. (MGA) A Good Stock To Buy Now?',
  'description': None},
 {'title': 'The Best History Trivia Quiz Questions to See How Clever You Really Are',
  'description': 'Test your knowledge of history—from ancient civilizations to World War I and II and beyond—with these 100 history trivia questions.'},
 {'title': 'RealClear Goes to London',
  'description': 'RealClear is vacationing in London this June, and we invite you to join us.When? June 23-25.Why? London is being overtaken by over four thousand civilized souls, from over one h

In [ ]:
result = fetch_supplier_news("Bosch")
print(f"Found {len(result)} articles")
if result:
    print(result[0])

In [4]:
## Scoring workflow
from typing import List, TypedDict
import json
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


# 1. Define exactly what we want our final JSON tool output to look like
class SupplierRiskReport(TypedDict):
    supplier: str
    score: int
    reasoning: str
    risk_factors: List[str]



def score_supplier_risk(supplier_name: str) -> dict:
    articles = fetch_supplier_news(supplier_name)
    articles_text = "\n\n".join(
        [f"Title: {a['title']}\nDescription: {a['description']}" for a in articles]
    )
# Call the LLM to get a risk score and reasoning
    prompt = ChatPromptTemplate.from_template(
     """You are a Supply Chain Risk Analyst specialising in the automotive industry.

    You have been given {num_articles} recent news articles about the supplier "{supplier_name}".

    Articles:
    {articles_text}

    Based on these articles, evaluate the supplier's risk profile and return a JSON object with exactly these keys:
    - "supplier": the supplier name
    - "score": an integer from 1 to 10 where 1 is no risk and 10 is critical risk
    - "reasoning": one sentence explaining the score
    - "risk_factors": a list of strings, each describing one specific risk identified

    Return JSON only. No explanation, no markdown, no code blocks."""
)
# Initialize the model
    base_llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)

#strctutured json out from basellm
    structured_llm =base_llm.with_structured_output(SupplierRiskReport)

    chain = prompt | structured_llm

    result = chain.invoke({
        "supplier_name": supplier_name,
        "num_articles": len(articles),
        "articles_text": articles_text
    })

    return result

final_report = score_supplier_risk("Bosch")
print(final_report)

{'supplier': 'Bosch', 'score': 1, 'reasoning': 'The articles are unrelated consumer/media mentions and a retail product deal, with no indication of financial, operational, regulatory, or automotive supply disruption risk for Bosch.', 'risk_factors': ['No news evidence of production disruption or supply shortage', 'No indication of recalls, quality issues, or regulatory action', "Articles appear unrelated to Bosch's automotive supplier operations", 'Duplicate and promotional content suggests low relevance to supplier risk']}


In [5]:
import time

suppliers = ["Bosch", "Aptiv", "Magna International", "Samsung SDI", "BASF"]
all_reports = []

for supplier in suppliers:
    report = score_supplier_risk(supplier)
    all_reports.append(report)
    print(f"{report['supplier']}: {report['score']}/10 — {report['reasoning']}")
    time.sleep(2)

Bosch: 1/10 — The provided articles are unrelated to Bosch's automotive supply operations and only indicate consumer/retail mentions, so there is no evidence of operational, financial, or supply chain risk.
Aptiv: 3/10 — The articles indicate Aptiv is innovating and improving operational safety and market positioning, with no evidence of acute financial, legal, or supply disruption risk, so overall supplier risk appears low to moderate.
Magna International: 2/10 — The recent articles are mostly unrelated or mildly positive analyst/stock commentary, with no evidence of operational disruption, financial distress, or supply chain issues for Magna International.
Samsung SDI: 5/10 — Samsung SDI appears strategically important and commercially active with major automaker wins, but project delays and dependence on large EV programs create moderate supply and execution risk.
BASF: 5/10 — BASF shows strategic growth and large-scale investment momentum, but the articles also point to environment

In [7]:
def build_risk_report(all_reports: list[dict]):
    """
    Sorts risk reports, extracts high-risk suppliers for an alert section,
    and prints the full report in markdown.
    """
    # 1. Sort all reports by score (highest first)
    sorted_reports = sorted(all_reports, key=lambda x: x['score'], reverse=True)
    
    # 2. Filter for High Risk (Score >= 5)
    high_risk_list = [r for r in sorted_reports if r['score'] >= 5]
    
    # 3. Print the Header
    print("# Supplier Risk Report — June 2026\n")
    
    # 4. Print HIGH RISK SECTION (if any exist)
    if high_risk_list:
        print("## ⚠️ HIGH RISK SUPPLIERS (Action Required)")
        for supplier in high_risk_list:
            print(f"- **{supplier['supplier']}**: Score {supplier['score']} - {supplier['reasoning']}")
        print("\n---\n") # Adds a horizontal line for clean separation

    # 5. Print the Full Markdown Table
    print("## All Supplier Assessments")
    print("| Supplier | Risk Score | Risk Factors | Reasoning |")
    print("| :--- | :--- | :--- | :--- |")
    
    for report in sorted_reports:
        factors_str = ", ".join(report['risk_factors'])
        print(f"| {report['supplier']} | {report['score']} | {factors_str} | {report['reasoning']} |")


build_risk_report(all_reports)

# Supplier Risk Report — June 2026

## ⚠️ HIGH RISK SUPPLIERS (Action Required)
- **Samsung SDI**: Score 5 - Samsung SDI appears strategically important and commercially active with major automaker wins, but project delays and dependence on large EV programs create moderate supply and execution risk.
- **BASF**: Score 5 - BASF shows strategic growth and large-scale investment momentum, but the articles also point to environmental liability exposure, regulatory scrutiny around PFAS, and macro-industrial weakness in Germany that could affect operational resilience.

---

## All Supplier Assessments
| Supplier | Risk Score | Risk Factors | Reasoning |
| :--- | :--- | :--- | :--- |
| Samsung SDI | 5 | Delayed or paused $3.5 billion EV battery joint project with GM in Indiana, indicating execution and schedule risk, Heavy exposure to large OEM programs such as Volkswagen and GM, increasing concentration risk if any program is postponed or cancelled, New battery supply commitments in Europe 

In [ ]:
def flag_high_risk(reports: list[dict]) -> list[dict]:
    return [r for r in reports if r['score'] >= 7]